# 1 · Nodes

A node is a function, and its type hints are its interface: each parameter is an input, and the return type is its output. This notebook writes a few nodes, shows what a person sees of them in an editor, and puts them in a registry.

In [ ]:
import json
from dataclasses import dataclass
from typing import Annotated, Literal

from conductor import Choice, NodeRegistry, Param, node
from conductor.widgets import Dropdown, Slider, Textarea, Widget

## One input, one output

The smallest node. Its one output is called `result`. A node is still its function, so calling it is its unit test.

In [ ]:
@node("shout", title="Shout", description="Turns a text into capital letters.")
def shout(text: str) -> str:
    return text.upper()


shout("hello")

## What a person reads

`Annotated[..., Param(...)]` says what a person sees of an input in an editor: its `title`, its `description` and its `widget`, the control the value is edited with.

A widget is optional. Without one, the host's editor picks a control from the input's type, such as a line of text for a `str`. So give a widget only where it says more than the type, such as a `Textarea` for a long text or a `Slider` for a number in a range.

`Param` also takes limits, such as `ge` and `le`, and every value an input gets is held to them.

In [ ]:
@node("truncate", title="Truncate", description="Cuts a text down to a length.", category="text")
def truncate(
    text: Annotated[str, Param(title="Text", widget=Textarea(rows=3))],
    max_length: Annotated[int, Param(title="Length", ge=1, le=500, widget=Slider(step=1))] = 100,
    add_ellipsis: Annotated[bool, Param(title="Add ...")] = True,
) -> Annotated[str, Param(title="Truncated")]:
    if len(text) <= max_length:
        return text
    truncated = text[:max_length]
    return f"{truncated}..." if add_ellipsis else truncated


truncate(text="The quick brown fox jumps over the lazy dog", max_length=19)

## A dropdown

`Param(choices=...)` names the values an input takes. Each choice is a value, or a `Choice` to give it a title a person reads, and the input gets the choice's `id`. `Dropdown()` draws them. A default makes an input optional.

In [ ]:
STYLES = (
    Choice("first-last", "First Last"),
    Choice("last-first", "Last, First"),
    Choice("upper-first", "LAST First"),
)


@node("format-name", title="Format name", description="Writes a full name in a style.", category="text")
def format_name(
    first: Annotated[str, Param(title="First name")],
    last: Annotated[str, Param(title="Last name")],
    style: Annotated[str, Param(title="Style", choices=STYLES, widget=Dropdown())] = "first-last",
) -> Annotated[str, Param(title="Full name")]:
    match style:
        case "last-first":
            return f"{last}, {first}"
        case "upper-first":
            return f"{last.upper()} {first}"
        case _:
            return f"{first} {last}"


format_name(first="Ada", last="Lovelace", style="upper-first")

## Several outputs

Return a frozen dataclass, and each of its fields is an output of that name. A `Param` on a field titles it.

In [ ]:
@dataclass(frozen=True)
class Lines:
    first: Annotated[str, Param(title="First line")]
    rest: Annotated[str, Param(title="Remaining")]


@node("split-text", title="Split text", description="Splits a text into its first line and the rest.", category="text")
def split_text(text: Annotated[str, Param(title="Text", widget=Textarea(rows=4))]) -> Lines:
    first, _, rest = text.partition("\n")
    return Lines(first=first, rest=rest)


split_text("hello\nworld\nagain")

## What an editor gets

`describe()` is a node as an editor's palette shows it. It is a pydantic model, so `model_dump(mode="json")` gives its JSON. Each input carries its type, its JSON schema with the limits in it, its title and its widget, if it has one.

In [ ]:
for port in truncate.describe().inputs:
    print(json.dumps(port.model_dump(mode="json", exclude_none=True)))

## A control of your own

`conductor.widgets` ships the common controls: `TextInput`, `Textarea`, `Dropdown`, `NumberInput`, `Slider`, `Switch`, `DatePicker`, `FileUpload` and `ListInput`. For any other, subclass `Widget` in your own code, next to the component that draws it. Conductor passes it on to the editor as JSON and never reads it.

In [ ]:
class EntityDropdown(Widget):
    """A list of things the host looks up, such as its users."""

    type: Literal["entity-dropdown"] = "entity-dropdown"
    entity_kind: str = ""


@node("assign", title="Assign", description="Names who owns a task.", category="text")
def assign(
    task: str,
    owner: Annotated[str, Param(title="Owner", widget=EntityDropdown(entity_kind="user"))],
) -> str:
    return f"{task}: {owner}"


assign.describe().inputs[1].widget

## The registry

`@node` registers nothing. A registry is the set of nodes a host offers. `registry.add(...)` puts nodes on it, and `@registry.node(...)` takes `@node`'s arguments and adds the node in one step. A node added to a registry takes the registry's defaults, here its category. `registry.describe()` is the whole palette.

In [ ]:
registry = NodeRegistry(category="text")
registry.add(shout, truncate, format_name, split_text, assign)


@registry.node("reverse", title="Reverse", description="Reverses a text.")
def reverse(text: str) -> str:
    return text[::-1]


for described in registry.describe().nodes:
    inputs = ", ".join(f"{port.name}: {port.type}" for port in described.inputs)
    outputs = ", ".join(f"{port.name}: {port.type}" for port in described.outputs)
    print(f"{described.id} ({described.category}): ({inputs}) -> ({outputs})")

## Registries from modules

A module can keep a registry of its own, with defaults for its nodes, and the host adds it with `add_subregistry`: its nodes are copied in, its tags first. A second node under one id raises. `filter` gives a narrower registry, say the nodes one customer may use. Its predicate gets each node's whole definition.

In [ ]:
beta = NodeRegistry(tags=["beta"])


@beta.node("whisper", title="Whisper", description="Turns a text into small letters.")
def whisper(text: str) -> str:
    return text.lower()


registry.add_subregistry(beta)
stable = registry.filter(lambda definition: "beta" not in definition.tags)
[definition.id for definition in registry], [definition.id for definition in stable]